# ACE-iPred — frozen model reproduction

Reproducibility notebook for the frozen revision analysis. It trains the Random Forest only from the packaged training data, computes SHAP values, and exports the fitted model. Dataset-audit and benchmark outputs used in the revision are supplied separately in the package.

In [ ]:
import pandas as pd, numpy as np, joblib, shap
from sklearn.ensemble import RandomForestClassifier
from pathlib import Path

In [ ]:
FEATURES = ['BLOSUM4', 'BLOSUM1', 'KF5', 'VHSE2', 'BLOSUM10', 'SVGER2', 'SVGER3', 'ProtFP7', 'SVGER4', 'MSWHIM3']
MODEL_PARAMS = {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 4, 'max_features': 1.0, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}
FEATURES, MODEL_PARAMS

In [ ]:
ROOT = Path.cwd().parent
path = ROOT / "data" / 'ACE_feat.xlsx'
xls = pd.ExcelFile(path)
print(xls.sheet_names)


In [ ]:
df = pd.read_excel(path)
labels = pd.read_excel(ROOT/"data"/"ACE_iPred_similarity_cleaned.xlsx", sheet_name="train_cleaned")
label_map = dict(zip(labels["seq"].astype(str).str.strip().str.upper(), labels["class"]))
df["_seq"] = df["seq"].astype(str).str.strip().str.upper()
y = df["_seq"].map(label_map)
assert y.notna().all(), "Some training sequences have no class label."
y = y.astype(int)
X = df[FEATURES].copy()
print(df.shape, y.value_counts().to_dict())

In [ ]:
rf = RandomForestClassifier(**MODEL_PARAMS)
rf.fit(X, y)
joblib.dump(rf, ROOT/"results"/"frozen_random_forest.joblib")
print("Saved fitted model.")

In [ ]:
explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(X)
arr = np.asarray(sv)
if isinstance(sv, list):
    sv_pos = np.asarray(sv[1])
elif arr.ndim == 3:
    sv_pos = arr[:, :, 1]
else:
    sv_pos = arr
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Mean_abs_SHAP": np.abs(sv_pos).mean(axis=0)
}).sort_values("Mean_abs_SHAP", ascending=False)
importance

In [ ]:
importance.to_excel(ROOT/"results"/"reproduced_SHAP_global_importance.xlsx", index=False)
print("Done.")